In [1]:
!pip install kafka-python

In [1]:
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers='host.docker.internal:9093',
    key_serializer=lambda k: k.encode('utf-8') if k else None,
    value_serializer=lambda v: v.encode('utf-8')
)

# Send value only
producer.send('my-topic', value='Hello Kafka!')
producer.flush()
print("Message sent!")


Message sent!


In [2]:
# Send key-value pair
producer.send('my-topic', key='order123', value='Order Placed')
producer.flush()
print("Key-value message sent!")

# Error handling example
try:
    producer.send('my-topic', value='Error Test')
    producer.flush()
except Exception as e:
    print(f"Error occurred: {e}")

Key-value message sent!


In [3]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    'my-topic',
    bootstrap_servers='host.docker.internal:9093',
    auto_offset_reset='earliest',
    enable_auto_commit=False,
    group_id='my-consumer-group',
    key_deserializer=lambda k: k.decode('utf-8') if k else None,
    value_deserializer=lambda v: v.decode('utf-8')
)

for message in consumer:
    print(
        f"Key: {message.key}, "
        f"Value: {message.value}"
    )
    consumer.commit()
    break


Key: order123, Value: Order Placed


In [4]:
from kafka import KafkaProducer
import time

producer = KafkaProducer(
    bootstrap_servers='host.docker.internal:9093',
    key_serializer=lambda k: k.encode('utf-8'),
    value_serializer=lambda v: v.encode('utf-8')
)

# Send messages
messages = [
    ('order1', 'Order Created'),
    ('order2', 'Order Paid'),
    ('order3', 'Order Shipped')
]

for key, value in messages:
    producer.send('my-topic', key=key, value=value)
    print(f"Sent -> Key: {key}, Value: {value}")
    time.sleep(0.5)

producer.flush()
producer.close()


Sent -> Key: order1, Value: Order Created
Sent -> Key: order2, Value: Order Paid
Sent -> Key: order3, Value: Order Shipped


In [5]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    'my-topic',
    bootstrap_servers='host.docker.internal:9093',
    auto_offset_reset='earliest',
    enable_auto_commit=False,
    group_id=None,  #  IMPORTANT: no group = always read all messages
    consumer_timeout_ms=5000,  #  exit after 5 seconds if no messages
    key_deserializer=lambda k: k.decode('utf-8') if k else None,
    value_deserializer=lambda v: v.decode('utf-8')
)

print("Receiving messages...\n")

for message in consumer:
    print(f"Received -> Key: {message.key}, Value: {message.value}")

consumer.close()
print("\nDone reading messages.")


Receiving messages...

Received -> Key: None, Value: Hello Kafka!
Received -> Key: order123, Value: Order Placed
Received -> Key: None, Value: Error Test
Received -> Key: None, Value: Hello Kafka!
Received -> Key: order123, Value: Order Placed
Received -> Key: None, Value: Error Test
Received -> Key: None, Value: Hello Kafka!
Received -> Key: order123, Value: Order Placed
Received -> Key: None, Value: Error Test
Received -> Key: order123, Value: Order Placed
Received -> Key: None, Value: Error Test
Received -> Key: None, Value: Hello Kafka!
Received -> Key: order123, Value: Order Placed
Received -> Key: None, Value: Error Test
Received -> Key: order1, Value: Order Created
Received -> Key: order2, Value: Order Paid
Received -> Key: order3, Value: Order Shipped

Done reading messages.
